# Cierre · Validación de la entrega y conclusiones

Este notebook verifica la coherencia entre las salidas de Python y los datos que consume Tableau. Las visualizaciones de negocio permanecen en el dashboard: aquí se presentan controles y conclusiones, sin crear gráficas ni exportar imágenes.

**Orden:** 01 → 02 → 03 → 04. **Entrada:** CSV de preparación y resultados. **Salida conservada:** `resultados/resumen.json`.

Los nombres de los notebooks organizan la ejecución; las fases de la guía académica son: comprensión, hipótesis, preparación, KPIs, dashboard y análisis/storytelling.

**Referencia del curso:** semanas 7–8, pp. 1–3 y 6. El cierre conecta indicadores, hallazgos y decisiones con alcance delimitado. Las tendencias temporales son condicionales en la guía: una sola captura no permite afirmarlas.


In [1]:
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

from bi_airbnb import (ARCHIVO_ANALITICO, DIR_RESULTADOS,
                       anfitriones_unicos, cargar_base, cargar_resultado,
                       exigir)

todo, d, hosts = cargar_base()
anuncios = pd.read_csv(DIR_RESULTADOS / "anuncios_dashboard.csv", dtype={"id_anuncio": str, "id_host": str})
anfitriones = pd.read_csv(DIR_RESULTADOS / "anfitriones_dashboard.csv", dtype={"id_host": str})
kpis = cargar_resultado("kpis.csv")
tests = cargar_resultado("hipotesis.csv")


## 1. Control de la exportación a Tableau
Se comprueban columnas, anuncios únicos y población exportada. Los prefijos `a_` y `h_` conservan los identificadores como texto. La limpieza y las claves del modelo ya se verifican en 01; aquí no se compara cada celda de nuevo.


In [2]:
columnas_esperadas = list(d.columns) + ["precio_intervalo_50mil", "activo_12m", "grupo_anfitrion"]
exigir(list(anuncios.columns) == columnas_esperadas, "Faltan columnas o cambió su orden en Tableau")
exigir(anuncios.id_anuncio.is_unique, "Hay anuncios duplicados en Tableau")
exigir(set(anuncios.id_anuncio) == set("a_" + d.id_anuncio), "La población exportada no coincide")
exigir(anfitriones.id_host.is_unique, "Hay anfitriones duplicados")
exigir(set(anuncios.id_host) == set(anfitriones.id_host), "Las claves de anfitrión no concilian")
print(f"Exportación verificada: {len(anuncios):,} anuncios, {len(anuncios.columns)} columnas y {len(anfitriones):,} anfitriones.")


Exportación verificada: 17,998 anuncios, 51 columnas y 7,795 anfitriones.


## 2. Conciliación independiente de los KPIs
Se comprueban los seis indicadores sobre el CSV realmente conectado a Tableau. Se validan valores y denominadores; K6 cuenta anfitriones distintos. Este control no sustituye abrir el libro y actualizar su extracto Hyper.


In [3]:
host_count = anuncios[["id_host", "n_anuncios"]].drop_duplicates()
valores = [anuncios.precio_cop.median(), anuncios.precio_por_huesped.median(),
           anuncios.calificacion.mean(), anuncios.resenas_12m.gt(0).sum() / anuncios.resenas_12m.count(),
           anuncios.ocupacion_estimada_pct.median(), host_count.n_anuncios.ge(2).mean()]
validos = [len(anuncios), anuncios.precio_por_huesped.count(), anuncios.calificacion.count(),
           anuncios.resenas_12m.count(), anuncios.ocupacion_estimada_pct.count(), len(host_count)]
exigir(kpis.id.is_unique and set(kpis.id) == {f"K{i}" for i in range(1, 7)}, "Lista de KPIs inválida")
control = kpis.set_index("id").loc[[f"K{i}" for i in range(1, 7)]].copy()
control["valor_desde_dashboard"] = valores
control["n_desde_dashboard"] = validos
control["coincide"] = np.isclose(control.valor, valores, rtol=1e-10, atol=1e-12) & control.n_validos.eq(validos)
exigir(control.coincide.all(), "Los KPIs no coinciden con el CSV del dashboard")
display(control[["indicador", "valor", "valor_desde_dashboard", "n_validos", "coincide"]])


,indicador,valor,valor_desde_dashboard,n_validos,coincide
id,,,,,
K1,Precio mediano por noche,160000.000000,160000.000000,17998,True
K2,Precio mediano por huésped,60000.000000,60000.000000,17998,True
K3,Calificación promedio,4.704137,4.704137,14549,True
K4,Anuncios con reseñas en 12 meses,0.671797,0.671797,17998,True
K5,Ocupación estimada mediana,0.065753,0.065753,17998,True
K6,Anfitriones con varios anuncios,0.305067,0.305067,7795,True


## 3. Resultados de las correlaciones
Se reutilizan H3 y H4 de 03. La magnitud y dirección se describen con rho de Spearman, junto con n y p.


In [4]:
display(tests[["hipotesis", "variable", "n", "rho", "p_presentacion", "decision"]])


,hipotesis,variable,n,rho,p_presentacion,decision
0,H3,capacidad,17998,0.480350,"< 0,001",Rechazar H0
1,H4,calificacion,14549,0.046257,"< 0,001",Rechazar H0


## 4. Conclusiones y decisiones
- Las tablas por localidad y alojamiento describen precios publicados de esta captura; no prueban diferencias poblacionales.
- Interpretar la dirección y magnitud de rho junto con el tamaño de muestra. Una correlación pequeña puede tener p < 0,05; no implica causalidad ni permite predecir un precio óptimo.
- Las medias y medianas por anfitrión son comparaciones descriptivas. No prueban ventajas de la gestión profesional.
- Usar los filtros de Tableau para comparar segmentos semejantes. No se observan reservas verificadas, costos ni rentabilidad; tampoco una evolución temporal.

**Presentación (semanas 7–8):** pregunta → KPI y segmento → hallazgo observado → decisión posible → limitación. Verificar los seis KPIs y filtros en Tableau y actualizar el extracto si cambian los datos.


## 5. Resumen de control
Conteos, fecha, trazabilidad y ejemplo de filtro.


In [5]:
chapinero = d[d.localidad.eq("Chapinero")]
calificados = int(d.calificacion.notna().sum())

resumen = {
    "anuncios_originales": len(todo), "anuncios_analizados": len(d),
    "anfitriones_analizados": len(hosts), "calificaciones_validas": calificados,
    "calificaciones_faltantes": int(len(d) - calificados),
    "anuncios_activos_12m": int(d.resenas_12m.gt(0).sum()),
    "hosts_varios": int(anfitriones_unicos(d).n_anuncios.ge(2).sum()),
    "fecha_captura": str(d.fecha_captura.iloc[0]),
    "sha256_entrada": hashlib.sha256(ARCHIVO_ANALITICO.read_bytes()).hexdigest(),
    "control_filtro_chapinero": {"n": len(chapinero),
                                 "precio_mediano": float(chapinero.precio_cop.median()),
                                 "calificacion_promedio": float(chapinero.calificacion.mean())},
}
(DIR_RESULTADOS / "resumen.json").write_text(
    json.dumps(resumen, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
resumen


{'anuncios_originales': 19505,
 'anuncios_analizados': 17998,
 'anfitriones_analizados': 7795,
 'calificaciones_validas': 14549,
 'calificaciones_faltantes': 3449,
 'anuncios_activos_12m': 12091,
 'hosts_varios': 2378,
 'fecha_captura': '2026-09-16',
 'sha256_entrada': 'eeead402c621028e1d942d5838138223538a356b91354790b4a1a7c38a0086eb',
 'control_filtro_chapinero': {'n': 4675,
  'precio_mediano': 190531.0,
  'calificacion_promedio': 4.692919909615868}}